# Notebook Q -- REGENERATE all Qwen2.5-VL-3B and Qwen2.5-VL-7B results at repetition_penalty = 1.0 (HEADLINE decoding)
Decision of the project owner, logged in `PREREG_MECHANISM.md` (fourth entry of 2026-11-06, written before this run): headline = plain greedy
decoding for all four models. The Qwen checkpoints ship `repetition_penalty = 1.05` in `generation_config.json`; every earlier Qwen result was produced with it
and becomes the **sensitivity analysis** (`_rp105`). This notebook produces the headline set. It does NOT overwrite anything (it runs in a fresh folder);
when merging locally the old Qwen files are renamed `*_rp105*` first.

How the penalty is switched off: the environment variable `CLOCK_REP_PENALTY=1.0` (read by `adapters.QwenVLAdapter.generate_answer`; unset = old behaviour)
for `text_only_v2`, `conversion_test`, `describe_check`, `eval_behavior`, and `--rep_penalty 1.0` for `mech_e3.py`. The run configs record both the
override and the checkpoint default.

Per model, in this order (most important first): text_only_v2 -> conversion_test -> describe_check -> eval_behavior -> E3e + E3f + text-light pilot
(`--no_gate`, no hooks; they read the describe/conversion files this very run just wrote, so they are all at 1.0). Qwen-3B first, cache purged, then Qwen-7B.
Every step is its own process; the output zip is refreshed after EVERY step, so it is safe to run in the background ("Save & Run All").

## Inputs
| Dataset | Contents at the dataset ROOT |
|---|---|
| **clock-code** (UPDATED version) | `adapters.py`, `clocks.py`, `eval_behavior.py`, `describe_check.py`, `text_only_v2.py`, `conversion_test.py`, `cap_check.py`, `mech_lib.py`, `mech_e3.py` |
| **clock-data** (the one notebooks A/B used) | folder `data/` (the ORIGINAL 500 clocks + `data.csv`) and folder `data_positions/` (480 clocks + `data.csv`) |

No clock-prior dataset is needed (the priors are regenerated here). **Settings:** GPU T4 x2, Internet ON, Save & Run All.
**Output:** `mech_Q_results.zip`. Runtime: a GUESS, about 6-7 h (3B ~2.5 h, 7B ~3.5-4 h; E3e/E3f/pilot cost was measured in D4: 21 and 33 min); no new step starts after 10.5 h.


In [ ]:
# CELL 1 -- GPU CHECK: stop the notebook right here if CUDA is not available
import torch, subprocess

MIN_GPUS = 2
if not torch.cuda.is_available():
    raise RuntimeError("STOP: CUDA is not available. Settings -> Accelerator -> 'GPU T4 x2', then Save & Run All again.")
n_gpu = torch.cuda.device_count()
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda:{i}  {p.name}  {p.total_memory / 2**30:.1f} GiB")
if n_gpu < MIN_GPUS:
    raise RuntimeError(f"STOP: this notebook needs {MIN_GPUS} GPU(s) but only {n_gpu} visible. Choose 'GPU T4 x2'.")
subprocess.run(["nvidia-smi"])
print("GPU check passed.")


In [ ]:
# CELL 2 -- COPY INPUTS from the attached Kaggle datasets into a clean working folder
import os, shutil, sys, time, json, subprocess, zipfile, datetime, hashlib

T_START = time.time()
INPUT, WORK = "/kaggle/input", "/kaggle/working"
RUN = f"{WORK}/run"
OUT_ZIP = f"{WORK}/mech_Q_results.zip"
CODE_FILES = ["adapters.py", "clocks.py", "eval_behavior.py", "describe_check.py", "text_only_v2.py", "conversion_test.py", "cap_check.py", "mech_lib.py", "mech_e3.py"]
EXPECT_N_DATA, EXPECT_N_POS = 500, 480          # images in data/ and data_positions/

shutil.rmtree(RUN, ignore_errors=True)
os.makedirs(f"{RUN}/logs")

def find_root(marker, maxdepth=4):
    """Directory under /kaggle/input that contains `marker` (a relative path). Dataset slugs don't matter."""
    base = INPUT.rstrip("/").count("/")
    for dirpath, dirnames, _ in os.walk(INPUT):
        if dirpath.count("/") - base > maxdepth:
            dirnames[:] = []
            continue
        if os.path.exists(os.path.join(dirpath, marker)):
            return dirpath
    raise FileNotFoundError(f"Could not find '{marker}' under {INPUT}. Attach the dataset that contains it "
                            f"(see the checklist at the top of this notebook).")

code_root = find_root("adapters.py")
for f in CODE_FILES:
    shutil.copy(os.path.join(code_root, f), RUN)
print("code from:", code_root)

for name, expect in (("data", EXPECT_N_DATA), ("data_positions", EXPECT_N_POS)):
    root = find_root(f"{name}/data.csv")
    shutil.copytree(os.path.join(root, name), f"{RUN}/{name}")
    n_png = len([x for x in os.listdir(f"{RUN}/{name}") if x.endswith(".png")])
    sha = hashlib.sha256(open(f"{RUN}/{name}/data.csv", "rb").read()).hexdigest()[:12]
    print(f"{name}/: {n_png} PNGs (expected {expect}), data.csv sha256[:12]={sha}, from {root}")
    assert n_png == expect, f"{name}/ has {n_png} PNGs, expected {expect} -- wrong or incomplete dataset"
print("inputs copied OK.")
du = shutil.disk_usage("/"); print(f"disk: {du.free / 2**30:.1f} GiB free of {du.total / 2**30:.1f} GiB")


In [ ]:
# CELL 3 -- ENVIRONMENT: env vars, versions, internet check (model weights are downloaded from Hugging Face)
os.environ.update({"PYTHONUNBUFFERED": "1", "TQDM_MININTERVAL": "60", "HF_HUB_DISABLE_PROGRESS_BARS": "1",
                   "TOKENIZERS_PARALLELISM": "false", "PYTHONIOENCODING": "utf-8", "PYTHONUTF8": "1",
                   "CLOCK_REP_PENALTY": "1.0"})      # HEADLINE decoding: plain greedy (the Qwen adapter reads this)
r = subprocess.run([sys.executable, "-c",
    "import torch, transformers, accelerate, torchvision, matplotlib, pandas, PIL, sys;"
    "print('python', sys.version.split()[0]);"
    "[print(m.__name__, m.__version__) for m in (torch, transformers, accelerate, torchvision, matplotlib, pandas, PIL)];"
    "print('cuda', torch.version.cuda, '| gpus', torch.cuda.device_count())"],
    capture_output=True, text=True)
print(r.stdout, r.stderr[-800:])
assert r.returncode == 0, "a required package is missing -- see the error above"
open(f"{RUN}/env.txt", "w").write(r.stdout + "\nstarted " + datetime.datetime.utcnow().isoformat() + "Z\n")
import urllib.request
try:
    urllib.request.urlopen("https://huggingface.co", timeout=20)
    print("internet OK (huggingface.co reachable)")
except Exception as e:
    raise RuntimeError(f"STOP: huggingface.co not reachable ({e}). Settings -> Internet -> ON.")


In [ ]:
# CELL 4 -- THE STEPS: (name, command, guessed_minutes)
M3B, A3B = "Qwen/Qwen2.5-VL-3B-Instruct", "qwen2.5-vl-3b"
M2B, A2B = "OpenGVLab/InternVL3-2B-hf", "internvl3-2b"
M7B, A7B = "Qwen/Qwen2.5-VL-7B-Instruct", "qwen2.5-vl-7b"
M8B, A8B = "OpenGVLab/InternVL3-8B-hf", "internvl3-8b"
DEADLINE_H = 10.5      # do not START a new step after this many hours
HARD_END_H = 11.5      # do not start a step whose guessed duration would end after this

def e3(model_id, adapter, parts, extra=()):
    return ["python", "mech_e3.py", "--model_id", model_id, "--adapter", adapter, "--parts", parts, *extra]

def purge_cache(*repos):
    code = ("import shutil,os,glob;"
            "[shutil.rmtree(p, ignore_errors=True) for r in %r for p in glob.glob(os.path.expanduser('~/.cache/huggingface/hub/models--'+r))]" % (list(repos),))
    return ["python", "-c", code]

NG = ("--no_gate",)
RP = ("--no_gate", "--rep_penalty", "1.0")
def beh(m_id, short):     # the four behaviour scripts; CLOCK_REP_PENALTY=1.0 is inherited from the environment (cell 3)
    ma = ["--model_id", m_id, "--adapter", short]
    return [("text_only_v2", ["python", "text_only_v2.py", *ma]),
            ("conversion_test", ["python", "conversion_test.py", *ma]),
            ("describe_check", ["python", "describe_check.py", "--data_csv", "data_positions/data.csv", "--images_dir", "data_positions", *ma]),
            ("eval_behavior", ["python", "eval_behavior.py", "--data_csv", "data/data.csv", "--images_dir", "data", *ma])]
EST3 = {"text_only_v2": 40, "conversion_test": 40, "describe_check": 30, "eval_behavior": 15}      # GUESSES
EST7 = {"text_only_v2": 60, "conversion_test": 60, "describe_check": 45, "eval_behavior": 20}      # GUESSES
STEPS = []
for i, (nm, cmd) in enumerate(beh(M3B, A3B)):
    STEPS.append((f"{i+1:02d}_q3b_{nm}_rp1", cmd, EST3[nm]))
STEPS += [("05_q3b_E3e_E3f_pilot_rp1", e3(M3B, A3B, "e,f,p", RP), 21),            # 21 min measured in D4
          ("06_purge_3b_cache",        purge_cache("Qwen--Qwen2.5-VL-3B-Instruct"), 1)]
for i, (nm, cmd) in enumerate(beh(M7B, A7B)):
    STEPS.append((f"{i+7:02d}_q7b_{nm}_rp1", cmd, EST7[nm]))
STEPS += [("11_q7b_E3e_E3f_pilot_rp1", e3(M7B, A7B, "e,f,p", RP), 33),            # 33 min measured in D4
          ("12_purge_7b_cache",        purge_cache("Qwen--Qwen2.5-VL-7B-Instruct"), 1)]
for name, cmd, est in STEPS:
    print(f"{name:<32} ~{est:>3} min (guess)  {' '.join(cmd)[:110]}")
print(f"guessed total: {sum(e for _, _, e in STEPS) / 60:.1f} h")


In [ ]:
# CELL 5 -- RUN every step. A failing step is recorded and the notebook CONTINUES with the next one.
EXPECTED = ['results/qwen2.5-vl-3b/text_only_v2.csv', 'results/qwen2.5-vl-3b/conversion_test.csv', 'results/qwen2.5-vl-3b/describe.csv', 'results/qwen2.5-vl-3b/results.csv', 'results/qwen2.5-vl-3b/run_config.json', 'results/qwen2.5-vl-3b/conversion_test_run_config.json', 'results/mech_e3/qwen2.5-vl-3b/e3e_transplant.csv', 'results/mech_e3/qwen2.5-vl-3b/e3f_forms.csv', 'results/mech_e3/qwen2.5-vl-3b/text_light_pilot.csv', 'results/qwen2.5-vl-7b/text_only_v2.csv', 'results/qwen2.5-vl-7b/conversion_test.csv', 'results/qwen2.5-vl-7b/describe.csv', 'results/qwen2.5-vl-7b/results.csv', 'results/qwen2.5-vl-7b/run_config.json', 'results/qwen2.5-vl-7b/conversion_test_run_config.json', 'results/mech_e3/qwen2.5-vl-7b/e3e_transplant.csv', 'results/mech_e3/qwen2.5-vl-7b/e3f_forms.csv', 'results/mech_e3/qwen2.5-vl-7b/text_light_pilot.csv']        # files that should exist at the end (relative to the run folder)
status = []

def make_zip():
    """Whitelisted snapshot of everything worth keeping; refreshed after EVERY step."""
    tmp = OUT_ZIP + ".tmp"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for sub in ("results", "logs"):
            for dp, _, fns in os.walk(f"{RUN}/{sub}"):
                for fn in fns:
                    full = os.path.join(dp, fn); z.write(full, os.path.relpath(full, RUN))
        for rel in ["status.json", "STATUS.txt", "env.txt", "conversion_images/manifest.csv",
                    ] + CODE_FILES:
            if os.path.exists(f"{RUN}/{rel}"):
                z.write(f"{RUN}/{rel}", rel)
    os.replace(tmp, OUT_ZIP)

def run_step(name, cmd, est_min):
    elapsed_h = (time.time() - T_START) / 3600
    if elapsed_h > DEADLINE_H or elapsed_h + est_min / 60 > HARD_END_H:
        return {"step": name, "status": "SKIPPED (time budget)", "returncode": None, "minutes": 0.0}
    cmd = [sys.executable if c == "python" else c for c in cmd]
    print(f"\n{'=' * 100}\n[{datetime.datetime.utcnow():%H:%M:%S}] START {name}  (elapsed {elapsed_h:.2f} h)\n{'=' * 100}", flush=True)
    t0 = time.time()
    with open(f"{RUN}/logs/{name}.log", "w", encoding="utf-8") as log:
        p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             encoding="utf-8", errors="replace", bufsize=1)
        for line in p.stdout:
            print(line, end="", flush=True); log.write(line)
        rc = p.wait()
    mins = (time.time() - t0) / 60
    print(f"[{name}] finished rc={rc} in {mins:.1f} min", flush=True)
    return {"step": name, "status": "OK" if rc == 0 else f"FAILED rc={rc}", "returncode": rc, "minutes": round(mins, 1)}

for name, cmd, est in STEPS:
    try:
        status.append(run_step(name, cmd, est))
    except Exception as e:           # never let one step's crash stop the notebook
        status.append({"step": name, "status": f"ERROR {type(e).__name__}: {e}"[:120], "returncode": None, "minutes": 0.0})
        print(f"[{name}] notebook-side error: {e}", flush=True)
    json.dump(status, open(f"{RUN}/status.json", "w"), indent=1)
    make_zip()                       # insurance: the zip is valid after every step
    print(f"zip refreshed after {name}: {os.path.getsize(OUT_ZIP) / 1e6:.1f} MB", flush=True)


In [ ]:
# CELL 6 -- FINAL: status table, completeness check, ONE output zip
lines = [f"{s['step']:<32} {s['status']:<26} {s['minutes']:>7} min" for s in status]
missing = [f for f in EXPECTED if not os.path.exists(f"{RUN}/{f}")]
lines += ["", f"total wall time: {(time.time() - T_START) / 3600:.2f} h",
          "MISSING expected files: " + (", ".join(missing) if missing else "none")]
open(f"{RUN}/STATUS.txt", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))
make_zip()
with zipfile.ZipFile(OUT_ZIP) as z:
    assert z.testzip() is None, "zip is corrupt"
    names = z.namelist()
print(f"\nOUTPUT: {OUT_ZIP}  ({os.path.getsize(OUT_ZIP) / 1e6:.1f} MB, {len(names)} files)")
for n in sorted(names):
    if n.startswith("results/") and n.endswith((".csv", ".json", ".png")):
        print("  ", n)
shutil.rmtree(RUN)      # leave ONLY the zip in /kaggle/working (the zip is verified above)
print("\nDONE." if not missing and all(s["status"] == "OK" for s in status) else
      "\nDONE WITH PROBLEMS -- read STATUS.txt inside the zip and the logs/ folder.")
